# 01 — Ingest Synergy API to the raw Volume

Pulls every entity in `synergy_schemas.ENTITIES` and lands each as JSON in the `raw_data` Volume,
partitioned by entity. Bronze (notebook 02) Auto Loads from here.

Each route is scoped on the axis the API actually supports. Dimensions are pulled in full so fact
foreign keys always resolve; facts are scoped by date and/or team.

- **Dimensions (full pull):** `teams`, `players`, `venues`, `umpires`, `leagues`, `divisions`,
  `conferences`, `competitions`. Empty-body `POST /api/<entity>/filter`, paged with skip/take.
  `players` is the full ~293k-player catalog — the API has no usable activity-date filter for it.
- **Date-windowed facts:** `games` (plus a `season` derived from the start-date year) and
  `practice_sessions`, windowed by `SYNERGY_START_DATE` / `SYNERGY_END_DATE`.
- **Practice sub-resources:** `practice/events` and `practice/training-workout`, by
  `practiceSessionIds` taken from the date-windowed `practice_sessions`.
- **Events:** the event routes scope by `teamIds` (from `SYNERGY_TEAM_IDS`) plus the date window.
  `teamIds` both limits events to your teams and lifts the API's 3-month date cap, which applies
  only when no player/team/game filter is given. Without `SYNERGY_TEAM_IDS` the event routes are
  skipped, since date-only events are an all-orgs firehose (~11k rows/day).
- **By team id:** `players_teamhistory` only accepts `teamId` (400 otherwise), pulled one id at a
  time from `SYNERGY_TEAM_IDS`. Skipped if unset.
- **Skipped:** `search_*`, which are typeahead duplicates of `players`/`teams`.

`SYNERGY_TEAM_IDS` scopes both events and teamhistory. No credentials yet? Use `tests/generate_fake_data`.

In [ ]:
# Job parameter bridge: when run as a Databricks Job task, parameters arrive as notebook widgets.
# Mirror them into os.environ before the setup cell reads them, so the same os.getenv(...) logic
# works as a job, in a workspace, or locally via .env. dbutils is absent under Databricks Connect,
# so the lookup is guarded.
import os
for _k in ("UC_CATALOG", "UC_SCHEMA", "SQL_WAREHOUSE_ID",
           "SYNERGY_START_DATE", "SYNERGY_END_DATE", "SYNERGY_TEAM_IDS"):
    try:
        _v = dbutils.widgets.get(_k)            # noqa: F821 — injected in Databricks
        if _v:
            os.environ[_k] = _v
    except Exception:
        pass

In [ ]:
# Dual-mode setup: runs locally via Databricks Connect and inside a Databricks notebook.
import os, json
try:
    from dotenv import load_dotenv; load_dotenv()
except Exception:
    pass
try:
    spark  # present inside a Databricks workspace notebook
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "main")
UC_SCHEMA     = os.getenv("UC_SCHEMA", "synergy_baseball_demo")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"
GOLD_SCHEMA   = f"{UC_SCHEMA}_gold"
VOLUME_NAME   = "raw_data"
VOLUME_PATH   = f"/Volumes/{UC_CATALOG}/{BRONZE_SCHEMA}/{VOLUME_NAME}"
B, S, G = f"{UC_CATALOG}.{BRONZE_SCHEMA}", f"{UC_CATALOG}.{SILVER_SCHEMA}", f"{UC_CATALOG}.{GOLD_SCHEMA}"
print("target:", B, "|", S, "|", G)

In [ ]:
def get_secret(env_var, scope="synergy", key=None):
    """Resolve a credential from .env first, then a Databricks secret scope, so the same notebook
    runs from a laptop or inside Databricks. Synergy creds live under scope `synergy`, keys
    `client_id` / `client_secret`. Never hardcode or commit them."""
    val = os.getenv(env_var)
    if val:
        return val
    from pyspark.dbutils import DBUtils  # type: ignore
    return DBUtils(spark).secrets.get(scope=scope, key=key or env_var.lower())

In [ ]:
import io
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

# Medallion schemas + raw Volume (idempotent).
for sch in [BRONZE_SCHEMA, SILVER_SCHEMA, GOLD_SCHEMA]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {UC_CATALOG}.{sch}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {UC_CATALOG}.{BRONZE_SCHEMA}.{VOLUME_NAME}")

def upload_json(volume_path: str, payload) -> int:
    """Write a JSON payload to the Volume via the Files API (no FUSE mount needed over Connect)."""
    body = json.dumps(payload).encode("utf-8")
    w.files.upload(volume_path, io.BytesIO(body), overwrite=True)  # SDK needs a seekable stream
    return len(body)
print("volume ready:", VOLUME_PATH)

In [ ]:
from synergy_client import SynergyAPI
from synergy_schemas import ENTITIES

# Auth: a pre-issued bearer token (SYNERGY_ACCESS_TOKEN) takes precedence if set. It expires after
# about an hour and can't be refreshed, so it suits a one-off pull. Otherwise use the OAuth
# client_id/client_secret flow, which the client re-mints automatically when the token expires.
access_token = os.getenv("SYNERGY_ACCESS_TOKEN")
if access_token:
    api = SynergyAPI(access_token=access_token)
    print("auth: using pre-issued SYNERGY_ACCESS_TOKEN")
else:
    api = SynergyAPI(
        client_id=get_secret("SYNERGY_CLIENT_ID", key="client_id"),
        client_secret=get_secret("SYNERGY_CLIENT_SECRET", key="client_secret"),
    )
    print("auth: using client_id/client_secret")

# Ingest window for the date-windowed facts (games, events, practice_sessions). The default is a
# narrow month so the all-orgs event routes stay tractable; widen it to capture more practice
# (sessions span 2024-2026). The games `season` is derived from the start year.
start  = os.getenv("SYNERGY_START_DATE", "2024-06-01")
end    = os.getenv("SYNERGY_END_DATE",   "2024-06-30")
season = int(start[:4])

def land(entity: str, rows: list, suffix: str = ""):
    path = f"{VOLUME_PATH}/{entity}/{entity}{('_' + suffix) if suffix else ''}.json"
    n = upload_json(path, rows)
    print(f"  {entity:28} {len(rows):>7,} rows ({n:,} bytes)")

In [ ]:
# One loop over the entity registry; the body/scoping differs by each entity's `scope`:
#   reference   -> pull all (empty body)        date       -> date window (+ season for games)
#   session     -> by practiceSessionIds        team_event -> teamIds + date window (event routes)
#   team        -> one teamId per id            skip       -> not ingested
# The team-keyed routes (team_event, team) use SYNERGY_TEAM_IDS and are skipped if it's unset.
# `practice_sessions` must precede the session routes in ENTITIES: it seeds session_ids.
from datetime import datetime
def _api_date(iso):                       # Synergy date filters want MM/DD/YYYY; ISO is silently ignored
    return datetime.strptime(iso, "%Y-%m-%d").strftime("%m/%d/%Y")

window      = {"minDate": _api_date(start), "maxDate": _api_date(end)}
team_ids    = [t.strip() for t in os.getenv("SYNERGY_TEAM_IDS", "").split(",") if t.strip()]
session_ids = []                          # captured from practice_sessions; used by the session routes

def suffix(scope):                        # filename suffix so date/team backfills don't overwrite each other
    if scope in ("date", "team_event"): return f"{start}_{end}"
    if scope in ("session", "team"):    return "scoped"
    return ""

for e in ENTITIES:
    name, scope = e["name"], e["scope"]
    if scope == "skip":
        continue
    if scope in ("team", "team_event") and not team_ids:
        print(f"  {name:28} skipped (set SYNERGY_TEAM_IDS)")
        continue
    try:
        if   scope == "reference":  rows = api.filter(e["path"], None)
        elif scope == "date":       rows = api.filter(e["path"], window | ({"season": season} if e.get("season") else {}))
        elif scope == "team_event": rows = api.filter(e["path"], {"teamIds": team_ids, **window})
        elif scope == "session":    rows = api.filter_by_ids(e["path"], "practiceSessionIds", session_ids, batch_size=50)
        elif scope == "team":       rows = [r for tid in team_ids for r in api.filter(e["path"], {"teamId": tid})]
        if name == "practice_sessions":
            session_ids = [r["id"] for r in rows if r.get("id")]   # seed the session routes below
        land(name, rows, suffix(scope))
    except Exception as ex:
        print(f"  {name:28} SKIP/err: {str(ex)[:70]}")

Re-run a different date window to backfill more: each window lands its own file, and the bronze Auto Loader picks up only new files. To scope to a single entity, filter `ENTITIES` above.